In [0]:
%sql
SELECT
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM workspace.supermarket.sales_transactions;

In [0]:
%sql
SELECT
    ROUND(
        SUM(total_amount) /
        NULLIF(COUNT(DISTINCT transaction_id), 0),
        2
    ) AS average_transaction_value
FROM workspace.supermarket.sales_transactions;

In [0]:
%sql
SELECT
    COUNT(DISTINCT product_id) AS unique_products_sold
FROM workspace.supermarket.sales_transactions;

In [0]:
%sql
SELECT
    store_name,
    total_revenue,
    transaction_count,
    average_transaction_value,
    unique_products,
    revenue_rank,
    transaction_rank
FROM workspace.supermarket.store_sales_summary
ORDER BY revenue_rank;

In [0]:
%sql
SELECT
    store_name,
    total_revenue,

    ROUND(
        100.0 * total_revenue /
        SUM(total_revenue) OVER (),
        2
    ) AS market_share_percent

FROM workspace.supermarket.store_sales_summary

ORDER BY market_share_percent DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.store_kpi_summary
USING DELTA
AS

WITH store_metrics AS (

    SELECT
        s.store_id,
        s.store_name,

        SUM(st.total_amount) AS total_revenue,

        COUNT(DISTINCT st.transaction_id)
            AS transaction_count,

        COUNT(DISTINCT st.product_id)
            AS unique_products,

        SUM(st.quantity_sold)
            AS total_units_sold

    FROM workspace.supermarket.sales_transactions st

    JOIN workspace.supermarket.stores s
        ON st.store_id = s.store_id

    GROUP BY
        s.store_id,
        s.store_name
),

with_metrics AS (

    SELECT
        *,

        total_revenue /
        NULLIF(transaction_count, 0)
        AS average_transaction_value

    FROM store_metrics
)

SELECT
    store_id,
    store_name,

    ROUND(total_revenue, 2)
        AS total_revenue,

    transaction_count,

    ROUND(
        average_transaction_value,
        2
    ) AS average_transaction_value,

    unique_products,

    total_units_sold,

    ROUND(
        100.0 * total_revenue /
        SUM(total_revenue) OVER (),
        2
    ) AS market_share_percent,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank,

    RANK() OVER (
        ORDER BY transaction_count DESC
    ) AS transaction_rank

FROM with_metrics;

In [0]:
%sql
SELECT *
FROM workspace.supermarket.store_kpi_summary
ORDER BY revenue_rank;

In [0]:
%sql
SELECT
    s.store_name,

    ROUND(
        SUM(st.quantity_sold) /
        NULLIF(
            AVG(i.quantity_on_hand),
            0
        ),
        2
    ) AS inventory_turnover_ratio

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.inventory i
    ON st.store_id = i.store_id
    AND st.product_id = i.product_id

JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id

GROUP BY
    s.store_name

ORDER BY
    inventory_turnover_ratio DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.store_inventory_turnover
USING DELTA
AS

SELECT
    s.store_id,
    s.store_name,

    ROUND(
        SUM(st.quantity_sold) /
        NULLIF(
            AVG(i.quantity_on_hand),
            0
        ),
        2
    ) AS inventory_turnover_ratio

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.inventory i
    ON st.store_id = i.store_id
    AND st.product_id = i.product_id

JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id

GROUP BY
    s.store_id,
    s.store_name;

In [0]:
%sql
SELECT *
FROM workspace.supermarket.store_inventory_turnover
ORDER BY inventory_turnover_ratio DESC;

In [0]:
%sql
SELECT
    p.category,

    SUM(st.quantity_sold)
        AS units_sold,

    ROUND(
        SUM(st.total_amount),
        2
    ) AS revenue

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.category

ORDER BY
    revenue DESC

LIMIT 3;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.category_sales_summary
USING DELTA
AS

SELECT
    p.category,

    SUM(st.quantity_sold)
        AS units_sold,

    ROUND(
        SUM(st.total_amount),
        2
    ) AS revenue

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.category;

In [0]:
%sql
SELECT
    YEAR(sale_date) AS year,
    ROUND(
        SUM(total_amount),
        2
    ) AS revenue
FROM workspace.supermarket.sales_transactions
GROUP BY YEAR(sale_date)
ORDER BY year;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.chain_kpi_summary
USING DELTA
AS

SELECT

    ROUND(
        SUM(total_amount),
        2
    ) AS total_revenue,

    COUNT(DISTINCT transaction_id)
        AS total_transactions,

    ROUND(
        SUM(total_amount) /
        NULLIF(
            COUNT(DISTINCT transaction_id),
            0
        ),
        2
    ) AS average_transaction_value,

    COUNT(DISTINCT product_id)
        AS unique_products_sold

FROM workspace.supermarket.sales_transactions;

In [0]:
%sql
SELECT *
FROM workspace.supermarket.chain_kpi_summary;